# Faktorizacijski model za recommendation system

U ovom zadatku koristim MovieLens 100K dataset kako bih implementirala recommendation sistem zasnovan na faktorizaciji matrice. Cilj je prikazati skup podataka, formirati sparse matricu, izvršiti faktorizaciju i rekonstruisati matricu sa predikcijama.

## Import biblioteka

U ovom koraku uvozim potrebne biblioteke koje će se koristiti za obradu podataka, rad sa matricama i faktorizaciju.

In [1]:
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
from sklearn.decomposition import TruncatedSVD

## Učitavanje skupa podataka

Učitavam MovieLens dataset koji sadrži ocjene korisnika za filmove. Svaki red predstavlja jednu ocjenu.

In [4]:
df = pd.read_csv(
    "u.data",
    sep="\t",
    names=["user_id", "item_id", "rating", "timestamp"]
)

Provjeravam dimenzije dataseta kako bih vidjela koliko ima redova i kolona.

In [5]:
df.shape

(100000, 4)

## Formiranje user-item matrice

Transformišem podatke u matricu gdje redovi predstavljaju korisnike, kolone filmove, a vrijednosti ocjene.
Na mjestima gdje nema ocjene pojavljuje se NaN.

In [6]:
user_item = df.pivot(index="user_id", columns="item_id", values="rating")

user_item.head()

item_id,1,2,3,4,5,6,7,8,9,10,...,1673,1674,1675,1676,1677,1678,1679,1680,1681,1682
user_id,,,,,,,,,,,,,,,,,,,,,
1,5.0,3.0,4.0,3.0,3.0,5.0,4.0,1.0,5.0,3.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,4.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Formiranje sparse matrice

Pošto većina vrijednosti nedostaje, matrica je rijetka (sparse). Zamjenjujem NaN sa 0 i pretvaram matricu u sparse format.

In [7]:
sparse_matrix = csr_matrix(user_item.fillna(0))

print("Shape:", sparse_matrix.shape)
print("Non-zero elementi:", sparse_matrix.nnz)
print("Ukupno elemenata:", sparse_matrix.shape[0] * sparse_matrix.shape[1])

Shape: (943, 1682)
Non-zero elementi: 100000
Ukupno elemenata: 1586126


In [8]:
density = sparse_matrix.nnz / (sparse_matrix.shape[0] * sparse_matrix.shape[1])
print("Gustina matrice:", density)
print("Sparsity:", 1 - density)

Gustina matrice: 0.06304669364224531
Sparsity: 0.9369533063577546


## Faktorizacija matrice

Primjenjujem SVD metodu kako bih razložila veliku matricu na dvije manje koje predstavljaju latentne faktore korisnika i filmova.

In [9]:
svd = TruncatedSVD(n_components=20, random_state=42)

user_factors = svd.fit_transform(sparse_matrix)
item_factors = svd.components_

print("User faktori shape:", user_factors.shape)
print("Item faktori shape:", item_factors.shape)

User faktori shape: (943, 20)
Item faktori shape: (20, 1682)


## Rekonstrukcija matrice

Nakon faktorizacije, množim matrice latentnih faktora korisnika i filmova kako bih dobila približnu verziju originalne user-item matrice.

Rekonstruisana matrica sadrži procijenjene vrijednosti i za one elemente gdje originalno nije postojala ocjena.

In [10]:
reconstructed = np.dot(user_factors, item_factors)

print(reconstructed.shape)
reconstructed[:5, :5]

(943, 1682)


array([[ 4.22801195,  2.09693674,  1.2766147 ,  3.13963673,  0.55145614],
       [ 2.0243085 , -0.00830876,  0.03383713,  0.27715874, -0.00817685],
       [-0.12239549, -0.06384159,  0.1698592 , -0.20550384, -0.09703686],
       [ 0.44911976, -0.1784593 ,  0.09267792, -0.07323029,  0.04139558],
       [ 3.69719876,  1.32220386,  0.35322062,  1.52484711,  0.50799794]])

## Poređenje originalne i rekonstruisane matrice

U ovom koraku upoređujem dio originalne matrice i rekonstruisane matrice.

Originalna matrica sadrži mnogo nula (nedostajućih vrijednosti), dok rekonstruisana matrica na tim mjestima daje procijenjene vrijednosti. Ove vrijednosti predstavljaju predikcije modela i mogu se koristiti za preporuke.

In [11]:
print("Originalna matrica:")
print(user_item.fillna(0).values[:5, :5])

print("\nRekonstruisana matrica:")
print(reconstructed[:5, :5])

Originalna matrica:
[[5. 3. 4. 3. 3.]
 [4. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [4. 3. 0. 0. 0.]]

Rekonstruisana matrica:
[[ 4.22801195  2.09693674  1.2766147   3.13963673  0.55145614]
 [ 2.0243085  -0.00830876  0.03383713  0.27715874 -0.00817685]
 [-0.12239549 -0.06384159  0.1698592  -0.20550384 -0.09703686]
 [ 0.44911976 -0.1784593   0.09267792 -0.07323029  0.04139558]
 [ 3.69719876  1.32220386  0.35322062  1.52484711  0.50799794]]


## Analiza dimenzija matrica

U ovom koraku prikazujem dimenzije originalne matrice, sparse matrice, kao i faktorizovanih matrica.

Može se vidjeti da je velika user-item matrica uspješno reducirana na dvije manje matrice sa latentnim faktorima, čime se postiže efikasnija reprezentacija podataka.

In [12]:
print("Original shape:", user_item.fillna(0).shape)
print("Sparse shape:", sparse_matrix.shape)
print("User factors shape:", user_factors.shape)
print("Item factors shape:", item_factors.shape)
print("Reconstructed shape:", reconstructed.shape)

Original shape: (943, 1682)
Sparse shape: (943, 1682)
User factors shape: (943, 20)
Item factors shape: (20, 1682)
Reconstructed shape: (943, 1682)


In [13]:
user_id = 0

predictions = reconstructed[user_id]

top_items = predictions.argsort()[-5:][::-1]

print("Top preporuke (item_id):", top_items)

Top preporuke (item_id): [ 49 167  99 173  88]


## Zaključak

U ovom radu prikazana je implementacija recommendation sistema zasnovanog na faktorizaciji matrice.

User-item matrica se pokazala kao vrlo rijetka (sparse), što je tipično za ovakve sisteme. Primjenom Truncated SVD metode matrica je razložena na latentne faktore korisnika i filmova.

Rekonstrukcijom matrice dobijene su procjene nedostajućih ocjena, što omogućava generisanje preporuka za korisnike.